# Módulo 3 — Explicabilidad con Grad-CAM y análisis clínico de errores
**TFM · Clasificación de lesiones cutáneas (HAM10000) con CNN y Grad-CAM**
Responsable: **Christian Daniel Zhirzhan Cabrera** · Objetivos específicos: **OE5, OE6**

### Entradas (del Módulo 2)
`TFM/results/representative_models.json` · `TFM/models/{arch}_seed{s}_best.keras` · `TFM/results/{arch}_seed{s}_test_preds.csv` · `TFM/results/mel_false_negatives.csv`

Si aún no existen, activar `DEMO_MODE = True` para desarrollar el pipeline con modelos ImageNet + cabeza aleatoria (los mapas **no** tienen significado clínico).

### Plan
1. Grad-CAM (y Grad-CAM++ opcional) sobre la última capa convolucional de cada arquitectura, usando los **logits** de la clase objetivo.
2. Comprobaciones de cordura (*sanity checks*): coherencia de la salida y test de aleatorización del modelo.
3. Mapas por clase (7 clases) en aciertos y errores, y comparación entre las 3 arquitecturas sobre las mismas imágenes.
4. Métricas cuantitativas *proxy*: energía del mapa dentro de la lesión, energía en el borde de la imagen, *pointing game*, acuerdo entre arquitecturas.
5. Detección de posibles atajos espurios (reglas, marcas de tinta, viñeteado, pelo, burbujas).
6. Falsos negativos de melanoma: ¿qué miró el modelo?
7. Plantilla de revisión cualitativa frente a criterios dermatoscópicos (regla ABCDE; la E — evolución — no es evaluable con una sola imagen).

In [ ]:
# @title 0. Entorno: Drive, repositorio y dependencias
# Requisitos (una sola vez por persona):
#  - En Drive: "Añadir acceso directo" de la carpeta compartida TFM a "Mi unidad".
#  - En Colab > Secretos (icono de llave): DAGSHUB_TOKEN y, si el repo es privado, GITHUB_TOKEN.
#  - Entorno de ejecución > Cambiar tipo > GPU T4.
import os, sys, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")

GITHUB_USER = "aReinoso007"
REPO_NAME = "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/TFM"   # carpeta compartida del grupo

try:
    _tok = userdata.get("GITHUB_TOKEN")
except Exception:
    _tok = None
_url = f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok else \
       f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", _url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, f"{REPO_DIR}/src")

!pip -q install "mlflow<3" dagshub kagglehub

import importlib, tfm_common as tc
importlib.reload(tc)
paths = tc.get_paths(DRIVE_ROOT)
print("Repo:", REPO_DIR, "| Drive:", paths.drive_root)

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Carga de datos y modelos

In [ ]:
DEMO_MODE = False          # True mientras el Módulo 2 no haya publicado modelos
CAM_METHODS = ["gradcam", "gradcampp"]

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, json
import tensorflow as tf
keras = tf.keras
sns.set_theme(style="white")
FIG_DIR = paths.drive_figures / "modulo3"; FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = paths.drive_results / "modulo3"; OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imágenes locales:", tc.ensure_local_images(paths))
df = tc.load_split(paths)
test_df = df[df.split == "test"].reset_index(drop=True)

In [ ]:
def load_model_fp32(arch, path=None):
    """Reconstruye el modelo en float32 y copia los pesos (evita problemas de
    precisión de los gradientes si se entrenó con mixed precision)."""
    m = tc.build_model(arch, weights=None if path else "imagenet", mixed_precision=False)
    if path:
        trained = keras.models.load_model(path, compile=False)
        m.set_weights(trained.get_weights())
    return m


def predict_df(model, arch, frame, bs=32):
    ds = tc.make_dataset(frame, arch, training=False, batch_size=bs)
    probs = model.predict(ds, verbose=0)
    out = frame[["image_id", "lesion_id", "dx", "label"]].copy().rename(columns={"dx": "dx_true", "label": "y_true"})
    out["y_pred"] = probs.argmax(1); out["dx_pred"] = [tc.CLASSES[i] for i in out.y_pred]
    for k, c in enumerate(tc.CLASSES): out[f"p_{c}"] = probs[:, k]
    return out


models, preds = {}, {}
if DEMO_MODE:
    for arch in tc.ARCHS:
        tc.set_seed(0)
        models[arch] = load_model_fp32(arch)          # ImageNet + cabeza aleatoria
        preds[arch] = predict_df(models[arch], arch, test_df)
else:
    rep = tc.load_json(paths.representative_json)
    for arch, info in rep.items():
        models[arch] = load_model_fp32(arch, info["model"])
        preds[arch] = pd.read_csv(info["test_preds"])
        # Comprobación: el modelo reconstruido reproduce las predicciones guardadas
        chk = predict_df(models[arch], arch, test_df.head(64))
        agree = (chk.y_pred.values == preds[arch].set_index("image_id").loc[chk.image_id].y_pred.values).mean()
        print(f"{arch}: seed {info['seed']} · concordancia con predicciones guardadas = {agree:.3f}")
for a, p in preds.items():
    p["correct"] = p.y_true == p.y_pred
    print(f"{a}: accuracy test = {p.correct.mean():.3f}")

## 2. Implementación de Grad-CAM y Grad-CAM++

In [ ]:
def build_cam_model(model):
    """Modelo auxiliar: entrada -> (activaciones de la última conv, logits)."""
    layer = tc.get_last_conv_layer_name(model)
    return keras.Model(model.inputs, [model.get_layer(layer).output, model.get_layer("head_logits").output]), layer


def compute_cams(cam_model, x, class_idx, method="gradcam"):
    """Mapas de activación para un batch.
    x: (B, H, W, 3) preprocesado · class_idx: (B,) clase objetivo por imagen.
    Grad-CAM (Selvaraju et al., 2017): pesos = media espacial de dL/dA.
    Grad-CAM++ (Chattopadhay et al., 2018): pesos con ponderación de órdenes superiores.
    Devuelve (B, IMG_SIZE, IMG_SIZE) normalizado a [0, 1]."""
    class_idx = tf.convert_to_tensor(class_idx, tf.int32)
    with tf.GradientTape() as tape:
        conv, logits = cam_model(x, training=False)
        score = tf.reduce_sum(logits * tf.one_hot(class_idx, tc.NUM_CLASSES), axis=1)
    grads = tape.gradient(score, conv)          # (B, h, w, C); muestras independientes
    if method == "gradcam":
        weights = tf.reduce_mean(grads, axis=(1, 2))
    elif method == "gradcampp":
        g2, g3 = grads ** 2, grads ** 3
        sum_a = tf.reduce_sum(conv, axis=(1, 2), keepdims=True)
        denom = 2.0 * g2 + sum_a * g3
        alphas = g2 / tf.where(tf.not_equal(denom, 0.0), denom, tf.ones_like(denom))
        weights = tf.reduce_sum(alphas * tf.nn.relu(grads), axis=(1, 2))
    else:
        raise ValueError(method)
    cam = tf.nn.relu(tf.reduce_sum(conv * weights[:, None, None, :], axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    cam = tf.image.resize(cam[..., None], (tc.IMG_SIZE, tc.IMG_SIZE), method="bilinear")[..., 0]
    return cam.numpy()


def load_batch(image_ids, arch):
    raw = np.stack([tc.load_image_uint8(str(paths.local_images / f"{i}.jpg")) for i in image_ids])
    return raw, tc.preprocess(tf.constant(raw, tf.float32), arch)


def cams_for(arch, image_ids, class_idx, method="gradcam", bs=16, cam_model=None):
    cm_ = cam_model if cam_model is not None else cam_models[arch][0]
    out = []
    for i in range(0, len(image_ids), bs):
        _, x = load_batch(image_ids[i:i + bs], arch)
        out.append(compute_cams(cm_, x, class_idx[i:i + bs], method))
    return np.concatenate(out) if out else np.zeros((0, tc.IMG_SIZE, tc.IMG_SIZE))


def overlay(ax, img, cam, title="", alpha=0.45):
    ax.imshow(img); ax.imshow(cam, cmap="jet", alpha=alpha, vmin=0, vmax=1)
    ax.set_title(title, fontsize=8); ax.axis("off")


cam_models = {a: build_cam_model(m) for a, m in models.items()}
for a, (_, layer) in cam_models.items():
    print(f"{a:15s} capa objetivo: {layer} (esperada: {tc.LAST_CONV_LAYER[a]})")

## 3. Comprobaciones de cordura

In [ ]:
# @title 3.1 Coherencia: softmax(logits del modelo auxiliar) == salida del modelo
ids = test_df.image_id.values[:8]
for a, m in models.items():
    _, x = load_batch(ids, a)
    _, logits = cam_models[a][0](x, training=False)
    diff = np.abs(tf.nn.softmax(logits).numpy() - m(x, training=False).numpy()).max()
    print(f"{a}: max |diferencia| = {diff:.2e}")

In [ ]:
# @title 3.2 Test de aleatorización del modelo (Adebayo et al., 2018)
# Si el mapa apenas cambia al aleatorizar pesos, la explicación no depende de lo aprendido.
from scipy.stats import spearmanr
import gc

def randomized_copy(arch, model, from_prefix):
    r = tc.build_model(arch, weights=None); r.set_weights(model.get_weights())
    base = [l for l in r.layers if not isinstance(l, keras.layers.InputLayer)]
    start = next(i for i, l in enumerate(base) if l.name.startswith(from_prefix))
    rng = np.random.default_rng(0)
    for l in base[start:]:
        if l.weights and not isinstance(l, keras.layers.BatchNormalization):
            l.set_weights([rng.normal(0, np.std(w) + 1e-3, w.shape).astype(w.dtype) for w in l.get_weights()])
    return r

sample = test_df.sample(min(32, len(test_df)), random_state=0)
rows = []
for a, m in models.items():
    cls = preds[a].set_index("image_id").loc[sample.image_id].y_pred.values
    orig = cams_for(a, sample.image_id.values, cls)
    for prefix in ["head_logits", tc.UNFREEZE_STAGES[a][0]]:
        r = randomized_copy(a, m, prefix)
        cm_r, _ = build_cam_model(r)
        rnd = cams_for(a, sample.image_id.values, cls, cam_model=cm_r)
        del r, cm_r; gc.collect()
        rho = [spearmanr(o[::8, ::8].ravel(), q[::8, ::8].ravel())[0] for o, q in zip(orig, rnd)]
        rows.append(dict(arch=a, aleatorizado_desde=prefix, spearman_medio=np.nanmean(rho)))
display(pd.DataFrame(rows).round(3))   # valores bajos = la explicación depende de los pesos aprendidos

## 4. Mapas por clase (OE5)

In [ ]:
# @title 4.1 Selección de casos: por clase, aciertos con mayor confianza y errores
K_CASES = 4
def select_cases(p, k=K_CASES):
    p = p.copy(); p["conf"] = p[[f"p_{c}" for c in tc.CLASSES]].max(axis=1)
    sel = []
    for c in tc.CLASSES:
        sub = p[p.dx_true == c]
        sel.append(sub[sub.correct].nlargest(k, "conf").assign(caso="acierto"))
        sel.append(sub[~sub.correct].nlargest(k, "conf").assign(caso="error"))
    return pd.concat(sel)

cases = {a: select_cases(p) for a, p in preds.items()}

In [ ]:
# @title 4.2 Galería por clase para cada arquitectura (clase objetivo = clase predicha)
for a in tc.ARCHS:
    cs = cases[a]
    for caso in ["acierto", "error"]:
        fig, axes = plt.subplots(len(tc.CLASSES), K_CASES * 2, figsize=(2.0 * K_CASES * 2, 2.1 * len(tc.CLASSES)))
        for r, c in enumerate(tc.CLASSES):
            sub = cs[(cs.dx_true == c) & (cs.caso == caso)]
            raws, _ = load_batch(sub.image_id.values, a) if len(sub) else (np.zeros((0,)), None)
            maps = cams_for(a, sub.image_id.values, sub.y_pred.values) if len(sub) else []
            for j in range(K_CASES):
                ax0, ax1 = axes[r, 2 * j], axes[r, 2 * j + 1]
                if j < len(sub):
                    row = sub.iloc[j]
                    ax0.imshow(raws[j]); ax0.axis("off")
                    ax0.set_title(f"{c} → {row.dx_pred}", fontsize=8)
                    overlay(ax1, raws[j], maps[j], f"p={row[f'p_{row.dx_pred}']:.2f}")
                else:
                    ax0.axis("off"); ax1.axis("off")
        fig.suptitle(f"Grad-CAM · {a} · {caso}s por clase (real → predicha)", y=1.0)
        fig.tight_layout(); fig.savefig(FIG_DIR / f"galeria_{a}_{caso}s.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# @title 4.3 Mismas imágenes en las 3 arquitecturas (Grad-CAM y Grad-CAM++)
common = test_df.groupby("dx").sample(1, random_state=3)       # una por clase
ncol = 1 + len(tc.ARCHS) * len(CAM_METHODS)
fig, axes = plt.subplots(len(common), ncol, figsize=(2.3 * ncol, 2.3 * len(common)))
for r, row in enumerate(common.itertuples()):
    raw = tc.load_image_uint8(row.path)
    axes[r, 0].imshow(raw); axes[r, 0].set_title(f"real: {row.dx}", fontsize=8); axes[r, 0].axis("off")
    col = 1
    for a in tc.ARCHS:
        pred = int(preds[a].set_index("image_id").loc[row.image_id].y_pred)
        for meth in CAM_METHODS:
            cam = cams_for(a, np.array([row.image_id]), np.array([pred]), meth)[0]
            overlay(axes[r, col], raw, cam, f"{a}\n{meth} · {tc.CLASSES[pred]}"); col += 1
fig.tight_layout(); fig.savefig(FIG_DIR / "comparativa_arquitecturas.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Métricas cuantitativas de localización (proxy)
HAM10000 no incluye máscaras de segmentación en este pipeline, así que se usa una **máscara aproximada de la lesión** (umbral de Otsu + componente conexa más central). Si se dispone de máscaras de referencia, sustituir `lesion_mask_proxy`.
- `energia_lesion`: fracción de la activación dentro de la máscara.
- `concentracion`: `energia_lesion / area_lesion` (> 1 = el mapa se concentra en la lesión más que al azar).
- `energia_borde`: fracción de activación en el 10 % exterior de la imagen (indicador de atajos: reglas, viñeteado, marcas).
- `pointing`: el máximo del mapa cae dentro de la lesión.

In [ ]:
from skimage.color import rgb2gray
from skimage.filters import gaussian, threshold_otsu
from skimage import morphology, measure
from scipy import ndimage as ndi

def lesion_mask_proxy(img):
    g = gaussian(rgb2gray(img), sigma=2)
    m = g < threshold_otsu(g)
    m = morphology.binary_opening(m, morphology.disk(3))
    lab = measure.label(m)
    if lab.max() == 0:
        return np.ones(img.shape[:2], bool)
    c = np.array(img.shape[:2]) / 2
    props = measure.regionprops(lab)
    # componente grande y cercana al centro (evita esquinas oscuras por viñeteado)
    best = max(props, key=lambda p: p.area / (1 + np.linalg.norm(np.array(p.centroid) - c)))
    return ndi.binary_fill_holes(lab == best.label)

FRAME = np.zeros((tc.IMG_SIZE, tc.IMG_SIZE), bool)
b = int(0.10 * tc.IMG_SIZE); FRAME[:b] = FRAME[-b:] = True; FRAME[:, :b] = FRAME[:, -b:] = True

def cam_metrics(cam, mask):
    s = cam.sum() + 1e-8
    e_les = float((cam * mask).sum() / s); area = float(mask.mean())
    y, x = np.unravel_index(cam.argmax(), cam.shape)
    return dict(energia_lesion=e_les, area_lesion=area, concentracion=e_les / max(area, 1e-3),
                energia_borde=float((cam * FRAME).sum() / s), pointing=bool(mask[y, x]))

# Control visual de las máscaras
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for j, row in enumerate(test_df.sample(8, random_state=1).itertuples()):
    img = tc.load_image_uint8(row.path); axes[0, j].imshow(img); axes[0, j].axis("off")
    axes[1, j].imshow(img); axes[1, j].contour(lesion_mask_proxy(img), colors="lime"); axes[1, j].axis("off")
fig.suptitle("Máscara proxy de la lesión (revisar calidad)"); fig.tight_layout(); plt.show()

In [ ]:
# @title 5.2 Métricas sobre todo el test para cada arquitectura y método (≈ minutos)
masks = {r.image_id: lesion_mask_proxy(tc.load_image_uint8(r.path)) for r in test_df.itertuples()}
rows, cam_store = [], {}
for a in tc.ARCHS:
    p = preds[a].set_index("image_id").loc[test_df.image_id]
    for meth in CAM_METHODS:
        maps = cams_for(a, test_df.image_id.values, p.y_pred.values, meth)
        cam_store[(a, meth)] = maps[:, ::4, ::4].astype(np.float16)   # 56x56 para acuerdo y guardado
        for iid, cam, yt, yp in zip(test_df.image_id.values, maps, p.y_true.values, p.y_pred.values):
            rows.append(dict(image_id=iid, arch=a, metodo=meth, dx_true=tc.CLASSES[yt], dx_pred=tc.CLASSES[yp],
                             correct=yt == yp, **cam_metrics(cam, masks[iid])))
loc = pd.DataFrame(rows)
loc.to_csv(OUT_DIR / "metricas_localizacion_test.csv", index=False)
np.savez_compressed(OUT_DIR / "cams_test_56px.npz", image_ids=test_df.image_id.values,
                    **{f"{a}__{m}": v for (a, m), v in cam_store.items()})

M = ["energia_lesion", "concentracion", "energia_borde", "pointing"]
display(loc.groupby(["arch", "metodo"])[M].mean().round(3))
display(loc[loc.metodo == "gradcam"].groupby(["arch", "correct"])[M].mean().round(3))
display(loc[loc.metodo == "gradcam"].pivot_table(index="dx_true", columns="arch", values="energia_lesion").round(3))

In [ ]:
# @title 5.3 Contrastes estadísticos
from scipy.stats import kruskal, mannwhitneyu
g = loc[loc.metodo == "gradcam"]
h, p = kruskal(*[g[g.arch == a].energia_lesion for a in tc.ARCHS])
print(f"Kruskal-Wallis energía en lesión entre arquitecturas: H={h:.2f}, p={p:.4f}")
for a in tc.ARCHS:
    s = g[g.arch == a]
    u, p = mannwhitneyu(s[s.correct].energia_borde, s[~s.correct].energia_borde)
    print(f"{a}: energía en borde, aciertos vs errores: p={p:.4f} "
          f"(medianas {s[s.correct].energia_borde.median():.3f} vs {s[~s.correct].energia_borde.median():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=g, x="arch", y="energia_lesion", hue="correct", ax=axes[0]); axes[0].set_title("Energía dentro de la lesión")
sns.boxplot(data=g, x="arch", y="energia_borde", hue="correct", ax=axes[1]); axes[1].set_title("Energía en el borde de la imagen")
fig.tight_layout(); fig.savefig(FIG_DIR / "metricas_localizacion.png", dpi=200); plt.show()

In [ ]:
# @title 5.4 Acuerdo entre arquitecturas (IoU del 20 % más activado y correlación de Spearman)
def top_mask(c, q=0.8): return c >= np.quantile(c, q)
rows = []
for i, a in enumerate(tc.ARCHS):
    for b2 in tc.ARCHS[i + 1:]:
        A, B = cam_store[(a, "gradcam")].astype(np.float32), cam_store[(b2, "gradcam")].astype(np.float32)
        iou = [ (top_mask(x) & top_mask(y)).sum() / ((top_mask(x) | top_mask(y)).sum() + 1e-8) for x, y in zip(A, B)]
        rho = [spearmanr(x.ravel(), y.ravel())[0] for x, y in zip(A, B)]
        rows.append(dict(par=f"{a} vs {b2}", IoU_top20=np.mean(iou), spearman=np.nanmean(rho)))
display(pd.DataFrame(rows).round(3))

## 6. Posibles atajos espurios (OE6)

In [ ]:
# @title 6.1 Imágenes con mayor activación fuera de la lesión / en el borde
for a in tc.ARCHS:
    top = loc[(loc.arch == a) & (loc.metodo == "gradcam")].nlargest(12, "energia_borde")
    fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
    maps = cams_for(a, top.image_id.values, np.array([tc.CLASS_TO_IDX[c] for c in top.dx_pred]))
    for ax, row, cam in zip(axes.ravel(), top.itertuples(), maps):
        overlay(ax, tc.load_image_uint8(str(paths.local_images / f"{row.image_id}.jpg")), cam,
                f"{row.dx_true}→{row.dx_pred} borde={row.energia_borde:.2f}")
    fig.suptitle(f"{a}: candidatos a atajo espurio (revisar artefactos)"); fig.tight_layout()
    fig.savefig(FIG_DIR / f"atajos_{a}.png", dpi=150); plt.show()

## 7. Falsos negativos de melanoma: ¿qué miró el modelo?

In [ ]:
# @title 7.1 Grad-CAM de la clase predicha vs Grad-CAM de 'mel' en los FN
fn_path = paths.drive_results / "mel_false_negatives.csv"
if fn_path.exists() and not DEMO_MODE:
    fn = pd.read_csv(fn_path)
    rep = tc.load_json(paths.representative_json)
    fn = fn[[r.seed == rep[r.arch]["seed"] for r in fn.itertuples()]]
else:
    fn = pd.concat([p[(p.y_true == tc.MEL_IDX) & (~p.correct)].assign(arch=a) for a, p in preds.items()])
    fn["n_modelos_fn"] = fn.groupby("image_id").arch.transform("count")

for a in tc.ARCHS:
    sub = fn[fn.arch == a].sort_values("n_modelos_fn", ascending=False).head(8)
    if sub.empty: continue
    fig, axes = plt.subplots(len(sub), 3, figsize=(7.5, 2.5 * len(sub)), squeeze=False)
    pred_idx = np.array([tc.CLASS_TO_IDX[c] for c in sub.dx_pred])
    cam_pred = cams_for(a, sub.image_id.values, pred_idx)
    cam_mel = cams_for(a, sub.image_id.values, np.full(len(sub), tc.MEL_IDX))
    for r, (row, cp, cmel) in enumerate(zip(sub.itertuples(), cam_pred, cam_mel)):
        img = tc.load_image_uint8(str(paths.local_images / f"{row.image_id}.jpg"))
        axes[r, 0].imshow(img); axes[r, 0].axis("off"); axes[r, 0].set_title(f"{row.image_id} · fallan {row.n_modelos_fn}", fontsize=8)
        overlay(axes[r, 1], img, cp, f"evidencia '{row.dx_pred}'")
        overlay(axes[r, 2], img, cmel, f"evidencia 'mel' (p={row.p_mel:.2f})")
    fig.suptitle(f"{a}: falsos negativos de melanoma"); fig.tight_layout()
    fig.savefig(FIG_DIR / f"fn_melanoma_{a}.png", dpi=150, bbox_inches="tight"); plt.show()

display(loc[(loc.metodo == "gradcam") & (loc.dx_true == "mel")].groupby(["arch", "correct"])[M].mean().round(3))

## 8. Plantilla de revisión cualitativa (criterios dermatoscópicos)
Se exporta una hoja para anotar, por imagen y arquitectura, si la activación se concentra en zonas relevantes para cada criterio. Columnas: `A_asimetria`, `B_borde`, `C_color`, `D_diametro_estructuras` (1 = sí, 0 = no, NA = no aplica), `artefacto` (ninguno / regla / tinta / pelo / viñeteado / burbuja / reflejo / otro) y `comentario`. La **E** (evolución) no puede evaluarse con una imagen única. Idealmente, revisión por un profesional sanitario; si no, documentarlo como limitación.

In [ ]:
N_PER_CLASS = 5
review = []
for a in tc.ARCHS:
    g = loc[(loc.arch == a) & (loc.metodo == "gradcam")]
    for c in tc.CLASSES:
        sub = g[g.dx_true == c]
        review.append(sub.sample(min(N_PER_CLASS, len(sub)), random_state=0))
review = pd.concat(review)[["image_id", "arch", "dx_true", "dx_pred", "correct", "energia_lesion", "energia_borde"]]
for col in ["A_asimetria", "B_borde", "C_color", "D_diametro_estructuras", "artefacto", "comentario"]:
    review[col] = ""
review.to_csv(OUT_DIR / "plantilla_revision_ABCD.csv", index=False)
print("Plantilla:", OUT_DIR / "plantilla_revision_ABCD.csv", len(review), "filas")

# Figuras individuales para la revisión
REV_DIR = FIG_DIR / "revision"; REV_DIR.mkdir(exist_ok=True)
for a in tc.ARCHS:
    sub = review[review.arch == a]
    maps = cams_for(a, sub.image_id.values, np.array([tc.CLASS_TO_IDX[c] for c in sub.dx_pred]))
    for row, cam in zip(sub.itertuples(), maps):
        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        img = tc.load_image_uint8(str(paths.local_images / f"{row.image_id}.jpg"))
        axes[0].imshow(img); axes[0].axis("off"); overlay(axes[1], img, cam, f"{a}: {row.dx_true}→{row.dx_pred}")
        fig.savefig(REV_DIR / f"{a}_{row.image_id}.png", dpi=120, bbox_inches="tight"); plt.close(fig)

## 9. Registro en MLflow

In [ ]:
mlflow = tc.init_mlflow()
with mlflow.start_run(run_name="modulo3_gradcam" + ("_demo" if DEMO_MODE else "")):
    mlflow.set_tags({"modulo": "3", "demo": DEMO_MODE})
    summ = loc.groupby(["arch", "metodo"])[M].mean()
    for (a, meth), r in summ.iterrows():
        tc.safe(mlflow.log_metrics, {f"{a}_{meth}_{k}": float(v) for k, v in r.items()})
    for f in ["metricas_localizacion_test.csv", "plantilla_revision_ABCD.csv"]:
        tc.safe(mlflow.log_artifact, str(OUT_DIR / f))
    tc.safe(mlflow.log_artifacts, str(FIG_DIR), "figuras")
print("Listo. Commit + push del notebook.")